In [ ]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *
from pyspark.sql.window import *

spark = (
    SparkSession.builder.master("local[*]").appName("user_interaction").getOrCreate()
)


data = [
    (1, 1001, 2002, "like", "2023-01-01 10:00:00"),
    (2, 1002, 1002, "comment", "2023-01-01 11:00:00"),
    (3, 1003, 2003, "share", "2023-01-02 10:00:00"),
    (4, 1004, 1004, "like", "2023-01-02 11:00:00"),
    (5, 1005, 2005, "comment", "2023-01-03 10:00:00"),
]

schema = ["interaction_id", "user1_id", "user2_id", "interaction_type", "timestamp"]

input_df = spark.createDataFrame(data, schema=schema)
input_df.show()

In [ ]:
win_spec = Window.partitionBy("user1_id").orderBy("user1_id")

input_df.filter(col("user1_id").isin(col("user2_id"))).withColumn(
    "self_interaction_count", row_number().over(win_spec)
).select("self_interaction_count", "user1_id").show()

+----------------------+--------+
|self_interaction_count|user1_id|
+----------------------+--------+
|                     1|    1002|
|                     1|    1004|
+----------------------+--------+

